# 使用 U-Net 进行图像分割

欢迎来到第三周的最后一份作业！你将构建自己的 U-Net，这是一种专为快速、精确的图像分割而设计的 CNN，并使用它来预测图像中每个像素的标签 - 在本例中是来自自动驾驶汽车数据集的图像。 

这种类型的图像分类称为语义图像分割。它与目标检测类似，都提出问题：“该图像中有哪些对象以及这些对象位于图像中的什么位置？”但目标检测用边界框标记对象，其中可能包括不属于对象的像素，而语义图像分割允许你通过使用其相应的类标记图像中的每个像素来预测图像中每个对象的精确掩码。这里的“语义”一词指的是所显示的内容，因此例如下面的“汽车”类由深蓝色掩码表示，“人”类由红色掩码表示：

<img src="images/carseg.png" style="width:500px;height:250px;">
<div style="text-align:center;"><u><b>图1</b></u>：分割图像示例<br></div>

正如你可能想象的那样，特定区域的标签对于自动驾驶汽车来说是一个非常重要的考虑因素，自动驾驶汽车需要对其环境有像素级的完美了解，以便它们可以改变车道并避开其他车辆，或任何可能危及人们生命的交通障碍。 

当你完成本 Notebook 时，你将能够： 

* 建立你自己的U-Net
* 解释一下普通CNN和U-net之间的区别
* 在 CARLA 自动驾驶汽车数据集上实现语义图像分割
* 应用稀疏分类交叉熵进行像素预测

前进，向着这伟大而光荣的追求！

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/convolutional-neural-networks/supplement/b6TbJ/optional-downloading-your-notebook-downloading-your-workspace-and-refreshing) 获取一份新的作业。

## 目录

- [1 - 依赖库](#1)
- [2 - 加载和分割数据](#2)
    - [2.1 - 将数据集分为原始图像和分割掩码](#2-1)
    - [2.2 - 预处理你的数据](#2-2)
- [3 - U-Net](#3)
    - [3.1 - 模型详细信息](#3-1)
    - [3.2 - 编码器（下采样块）](#3-2)
        - [练习 1 - conv_block](#ex-1)
    - [3.3 - 解码器（上采样块）](#3-3)
        - [练习 2 - upsampling_block](#ex-2)
    - [3.4 - 构建模型](#3-4)
        - [练习 3 - unet_model](#ex-3)
    - [3.5 - 设置模型尺寸](#3-5)
    - [3.6 - 损失函数](#3-6)
    - [3.7 - 数据集处理](#3-7)
- [4 - 训练模型](#4)
    - [4.1 - 创建预测掩码](#4-1)
    - [4.2 - 绘制模型准确率](#4-2)
    - [4.3 - 显示预测](#4-3)

<a name='1'></a>
## 1 - 依赖库

运行下面的单元格以导入你需要的所有库：

In [ ]:
### v1.2

In [ ]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dropout 
from tensorflow.keras.layers import Conv2DTranspose
from tensorflow.keras.layers import concatenate

from test_utils import summary, comparator

<a name='2'></a>
## 2 - 加载和分割数据

In [ ]:
import os
import numpy as np # 线性代数
import pandas as pd # 数据处理、CSV文件I/O (例如 pd.read_csv)

import imageio

import matplotlib.pyplot as plt
%matplotlib inline

path = ''
image_path = os.path.join(path, './data/CameraRGB/')
mask_path = os.path.join(path, './data/CameraMask/')
image_list_orig = os.listdir(image_path)
image_list = [image_path+i for i in image_list_orig]
mask_list = [mask_path+i for i in image_list_orig]

### 查看数据集中的部分原始图像与分割掩码：

完成探索后，返回`N=2`。否则 AutoGrader 将抛出 `list index out of range` 错误。

In [ ]:
N = 2
img = imageio.imread(image_list[N])
mask = imageio.imread(mask_list[N])
#mask = np.array([max(mask[i, j]) for i in range(mask.shape[0]) for j in range(mask.shape[1])]).reshape(img.shape[0], img.shape[1])

fig, arr = plt.subplots(1, 2, figsize=(14, 10))
arr[0].imshow(img)
arr[0].set_title('Image')
arr[1].imshow(mask[:, :, 0])
arr[1].set_title('Segmentation')

<a name='2-1'></a>
### 2.1 - 将数据集分为原始图像和分割掩码

In [ ]:
image_list_ds = tf.data.Dataset.list_files(image_list, shuffle=False)
mask_list_ds = tf.data.Dataset.list_files(mask_list, shuffle=False)

for path in zip(image_list_ds.take(3), mask_list_ds.take(3)):
    print(path)

In [ ]:
image_filenames = tf.constant(image_list)
masks_filenames = tf.constant(mask_list)

dataset = tf.data.Dataset.from_tensor_slices((image_filenames, masks_filenames))

for image, mask in dataset.take(1):
    print(image)
    print(mask)

<a name='2-2'></a>
### 2.2 - 预处理你的数据

通常，你的图像值除以 `255` 即可得到归一化。这将它们设置在 `0` 和 `1` 之间。但是，将 `tf.image.convert_image_dtype` 与 `tf.float32` 一起使用会将它们设置在 `0` 和 `1` 之间，因此无需将它们进一步除以 `255`。

In [ ]:
def process_path(image_path, mask_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=3)
    mask = tf.math.reduce_max(mask, axis=-1, keepdims=True)
    return img, mask

def preprocess(image, mask):
    input_image = tf.image.resize(image, (96, 128), method='nearest')
    input_mask = tf.image.resize(mask, (96, 128), method='nearest')

    return input_image, input_mask

image_ds = dataset.map(process_path)
processed_image_ds = image_ds.map(preprocess)

<a name='3'></a>
## 3 - U-Net 

U-Net，因其 U 形而得名，最初创建于 2015 年，用于肿瘤检测，但近年来已成为其他语义分割任务的非常流行的选择。 

U-Net 建立在称为全卷积网络 (FCN) 的先前架构之上，该架构用转置卷积层替换了典型 CNN 中的全连接层，该层将特征图上采样回原始输入图像的大小，同时保留空间信息。这是必要的，因为全连接层破坏了空间信息（图像的“位置”），而空间信息是图像分割任务的重要组成部分。使用转置卷积的另一个好处是输入大小不再需要固定，就像使用全连接层时那样。 

不幸的是，FCN 的最终特征层由于下采样过多而遭受信息丢失。在丢失如此多的信息后，上采样变得困难，导致输出看起来很粗糙。 

U-Net 对 FCN 进行了改进，使用了一些类似的设计，但在一些重要方面有所不同。  它不是在网络末端使用一个转置卷积，而是使用匹配数量的卷积将输入图像下采样到特征图，并使用转置卷积将这些图上采样回原始输入图像大小。它还添加了跳跃连接，以保留在编码过程中可能丢失的信息。 跳跃连接将信息从编码器中相应的下采样层发送到解码器中的每个上采样层，捕获更精细的信息，同时保持较低的计算量。这些有助于防止信息丢失以及模型过拟合。 

<a name='3-1'></a>
### 3.1 - 模型详细信息

<img src="images/unet.png" style="width:700px;height:400px;">
<div style="text-align:center;"><u><b> 图 2 </b></u>：U-Net 架构<br></div>

**收缩路径**（包含下采样步骤的编码器）：

图像首先通过几个卷积层，这些卷积层减少了高度和宽度，同时增加了通道数量。

收缩路径遵循常规的 CNN 架构，具有卷积层、激活值和池化层，用于对图像进行下采样并提取其特征。具体来说，它由重复应用两个 3 x 3 相同的填充卷积组成，每个卷积后跟一个修正线性单元 (ReLU) 和一个用于下采样的步长为 2 的 2 x 2 最大池化操作。在每个下采样步骤中，特征通道的数量都会加倍。

**裁剪函数**：此步骤从收缩路径中裁剪图像，并将其连接到扩展路径上的当前图像以创建跳跃连接。 

**扩展路径**（包含上采样步骤的解码器）：

扩展路径执行与收缩路径相反的操作，将图像恢复到原始大小，同时逐渐缩小通道。

具体来说，扩展路径中的每一步都会对特征图进行上采样，然后进行 2 x 2 卷积（转置卷积）。这种转置卷积将特征通道的数量减半，同时增加图像的高度和宽度。

接下来是与收缩路径中相应裁剪的特征图和两个 3 x 3 卷积的串联，每个卷积后跟一个 ReLU。你需要执行裁剪来处理每个卷积中边界像素的丢失。

**最终特征映射块**：最后一层使用 1x1 卷积，将每个 64 维特征向量映射为各类别的分数。卷积层的输出通道数等于卷积核数量，因此可以通过设置 1x1 卷积核的数量调整通道维度。在输出层中，使每个类别对应一个通道即可。

U-Net网络总共有23个卷积层。

#### 重要提示： 
作业中的 U-Net 示意图沿用原论文的层尺寸和卷积核配置，并使用较小的图像。受计算资源限制，本作业实际使用的卷积核数量只有图示的一半。保留原始配置图是为了帮助你了解原版 U-Net 的结构。关键规律是下采样时每经过一个阶段，卷积核数量翻倍。Notebook 中已提供实现本作业所需结构的全部说明和提示。

<a name='3-2'></a>
### 3.2 - 编码器（下采样块） 

<img src="images/encoder.png" style="width:500px;height:500px;">
<div style="text-align:center;"><u><b>图 3</b></u>：U-Net 编码器近距离 <br></div>

编码器是各种conv_blocks的堆栈：

每个 `conv_block()` 由 2 个 **Conv2D** 层组成，其中 ReLU 激活函数。我们会将 **Dropout** 和 **MaxPooling2D** 应用于某些 conv_blocks，你将在以下部分中进行验证，特别是下采样的最后两个块。 

函数将返回两个张量： 
- `next_layer`：这将进入下一个区块。 
- `skip_connection`：将进入相应的解码块。

**注意**：如果是`max_pooling=True`，则`next_layer`将是MaxPooling2D层的输出，但`skip_connection`将是之前应用的layer(Conv2D 或 Dropout，视情况而定)的输出。否则，两个结果将是相同的。  

<a name='ex-1'></a>
### 练习 1 - conv_block

实现`conv_block(...)`。以下是 `conv_block` 或合约区块中每个步骤的说明： 

* 添加 2 个 **Conv2D** 层，每层使用 `n_filters` 个卷积核；将 `kernel_size` 设为 3，`kernel_initializer` 设为 ['he_normal'](https://www.tensorflow.org/api_docs/python/tf/keras/initializers/HeNormal)，`padding` 设为 'same'，激活函数设为 'relu'。
* 如果 `dropout_prob` > 0，则添加参数为 `dropout_prob` 的 Dropout 层
* 如果 `max_pooling` 设置为 True，则添加一个池大小为 2x2 的 MaxPooling2D 层

In [ ]:
# UNQ_C1
# GRADED FUNCTION: conv_block
def conv_block(inputs=None, n_filters=32, dropout_prob=0, max_pooling=True):
    """
    卷积下采样块
    
    参数：
        input -- 输入张量
        n_filters -- 卷积层的卷积核数量
        dropout_prob -- Dropout 概率
        max_pooling -- 使用 MaxPooling2D 减少输出体的空间维度
    返回： 
        next_layer, skip_connection -- 下一层和跳跃连接输出
    """

    ### START CODE HERE
    conv = Conv2D(None, # 卷积核数量
                  None,   # 内核大小   
                  activation=None,
                  padding=None,
                  kernel_initializer='he_normal')(inputs)
    conv = Conv2D(None, # 卷积核数量
                  None,   # 内核大小
                  activation=None,
                  padding=None,
                  # 设置'kernel_initializer' 同上
                  kernel_initializer=None)(conv)
    ### END CODE HERE
    
    # 如果 dropout_prob > 0 添加 Dropout 层，并以变量 dropout_prob 作为参数
    if dropout_prob > 0:
         ### START CODE HERE
        conv = None
         ### END CODE HERE
         
        
    # 如果 max_pooling 为 True 添加一个带有 2x2 pool_size 的 MaxPooling2D
    if max_pooling:
        ### START CODE HERE
        next_layer = None
        ### END CODE HERE
        
    else:
        next_layer = conv
        
    skip_connection = conv
    
    return next_layer, skip_connection

In [ ]:
input_size=(96, 128, 3)
n_filters = 32
inputs = Input(input_size)
cblock1 = conv_block(inputs, n_filters * 1)
model1 = tf.keras.Model(inputs=inputs, outputs=cblock1)

output1 = [['InputLayer', [(None, 96, 128, 3)], 0],
            ['Conv2D', (None, 96, 128, 32), 896, 'same', 'relu', 'HeNormal'],
            ['Conv2D', (None, 96, 128, 32), 9248, 'same', 'relu', 'HeNormal'],
            ['MaxPooling2D', (None, 48, 64, 32), 0, (2, 2)]]

print('Block 1:')
for layer in summary(model1):
    print(layer)

comparator(summary(model1), output1)

inputs = Input(input_size)
cblock1 = conv_block(inputs, n_filters * 32, dropout_prob=0.1, max_pooling=True)
model2 = tf.keras.Model(inputs=inputs, outputs=cblock1)

output2 = [['InputLayer', [(None, 96, 128, 3)], 0],
            ['Conv2D', (None, 96, 128, 1024), 28672, 'same', 'relu', 'HeNormal'],
            ['Conv2D', (None, 96, 128, 1024), 9438208, 'same', 'relu', 'HeNormal'],
            ['Dropout', (None, 96, 128, 1024), 0, 0.1],
            ['MaxPooling2D', (None, 48, 64, 1024), 0, (2, 2)]]
           
print('\nBlock 2:')   
for layer in summary(model2):
    print(layer)
    
comparator(summary(model2), output2)

<a name='3-3'></a>
### 3.3 - 解码器（上采样块）

解码器或上采样模块将特征上采样回原始图像大小。在每个上采样级别，你将获取相应编码器块的输出并将其连接起来，然后再馈送到下一个解码器块。

<img src="images/decoder.png" style="width:500px;height:500px;">
<div style="text-align:center;"><u><b>图 4</b></u>：U-Net 解码器近距离 <br></div>

解码器中有两个新组件：`up` 和 `merge`。它们是转置卷积和跳跃连接。此外，还有两个卷积层设置为与编码器中相同的参数。 

在这里，你将遇到 `Conv2DTranspose` 层，它执行 `Conv2D` 层的逆操作。你可以阅读更多相关信息 [这里。](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2DTranspose)


<a name='ex-2'></a>
### 练习 2 - upsampling_block

实现`upsampling_block(...)`。

对于函数 `upsampling_block`： 
* 采用参数 `expansive_input`（来自前一层的输入张量）和 `contractive_input`（来自前一个跳过层的输入张量）
* 这里的卷积核数量与你之前完成的下采样块中的卷积核数量相同
* 你的 `Conv2DTranspose` 层将采用形状 (3,3) 和步长 (2,2) 的 `n_filters`，并将填充设置为 `same`。它应用于 `expansive_input`，或前一层的输入张量。 

此块也是你连接编码器块的输出的地方，创建跳跃连接。 

* 将 Conv2DTranspose 层的输出与收缩路径的输入沿 `axis` 为 3 的维度拼接。通常可以自行决定张量的拼接顺序，但为满足评分器要求，这里必须使用 `[up, contractive_input]`。

对于最终组件，将两个 Conv2D 层的参数设置为与编码器中为两个 Conv2D 层设置的值相同的值（ReLU 激活值，正常初始化器，`same` 填充）。 


In [ ]:
# UNQ_C2
# GRADED FUNCTION: upsampling_block
def upsampling_block(expansive_input, contractive_input, n_filters=32):
    """
    卷积上采样块
    
    参数：
        expansive_input -- 上一层的输入张量
        contractive_input -- 前一个跳过层的输入张量
        n_filters -- 卷积层的卷积核数量
    返回： 
        conv -- 张量输出
    """
    
    ### START CODE HERE
    up = Conv2DTranspose(
                 None,    # 卷积核数量
                 None,    # 内核大小
                 strides=None,
                 padding=None)(expansive_input)
    
    # 合并之前的输出和 contractive_input
    merge = concatenate([up, contractive_input], axis=3)
    conv = Conv2D(None,   # 卷积核数量
                 None,     # 内核大小
                 activation=None,
                 padding=None,
                 kernel_initializer='he_normal')(merge)
    conv = Conv2D(None,  # 卷积核数量
                 None,   # 内核大小
                 activation=None,
                 padding=None,
                  # 设置'kernel_initializer' 同上
                 kernel_initializer=None)(conv)
    ### END CODE HERE
    
    return conv

In [ ]:
input_size1=(12, 16, 256)
input_size2 = (24, 32, 128)
n_filters = 32
expansive_inputs = Input(input_size1)
contractive_inputs =  Input(input_size2)
cblock1 = upsampling_block(expansive_inputs, contractive_inputs, n_filters * 1)
model1 = tf.keras.Model(inputs=[expansive_inputs, contractive_inputs], outputs=cblock1)

output1 = [['InputLayer', [(None, 12, 16, 256)], 0],
            ['Conv2DTranspose', (None, 24, 32, 32), 73760],
            ['InputLayer', [(None, 24, 32, 128)], 0],
            ['Concatenate', (None, 24, 32, 160), 0],
            ['Conv2D', (None, 24, 32, 32), 46112, 'same', 'relu', 'HeNormal'],
            ['Conv2D', (None, 24, 32, 32), 9248, 'same', 'relu', 'HeNormal']]

print('Block 1:')
for layer in summary(model1):
    print(layer)

comparator(summary(model1), output1)

<a name='3-4'></a>
### 3.4 - 构建模型

在这里，你可以通过链接编码器、瓶颈和解码器将所有内容组合在一起！你需要指定输出通道的数量，对于该特定集合，输出通道的数量为 23。这是因为该自动驾驶汽车数据集中的每个像素有 23 个可能的标签。 

<a name='ex-3'></a>
### 练习 3 - unet_model

对于函数 `unet_model`，指定输入形状、卷积核数量和类数量（本例中为 23）。

对于模型的前半部分：

* 首先构建卷积块，将模型输入和卷积核数量传入该块。
* 然后，将每个块的第一个输出元素链接到下一个卷积块的输入
* 接下来，将每一步的卷积核数量加倍
* 从 `conv_block4` 开始，添加 `dropout_prob` 0.3
* 对于最终的conv_block，再次将`dropout_prob`设置为0.3，并关闭最大池化  

下半场：

* 使用cblock5作为expansive_input，cblock4作为contractive_input，`n_filters` * 8。这是你的瓶颈层。 
* 将前一个区块的输出链接为 expansive_input 和相应的收缩区块输出。
* 请注意，你必须在最大池化层之前使用收缩块的第二个元素。 
* 在每一步中，使用前一个块的卷积核数量的一半
* `conv9` 是一个带有 ReLU 激活值的 Conv2D 层，He 正常初始化器，`same` 填充
* 最后，使用 Conv2D 构建 `conv10`：卷积核数量等于类别数，卷积核大小为 1，填充方式为 same。`conv10` 的输出就是模型的输出。

In [ ]:
# UNQ_C3
# GRADED FUNCTION: unet_model
def unet_model(input_size=(96, 128, 3), n_filters=32, n_classes=23):
    """
    Unet 模型
    
    参数：
        input_size -- 输入形状 
        n_filters -- 卷积层的卷积核数量
        n_classes -- 输出类数
    返回： 
        model -- tf.keras.Model
    """
    inputs = Input(input_size)
    # 契约路径（编码）
    # 使用unet_模型和n_filters的输入添加conv_block
    ### START CODE HERE
    cblock1 = conv_block(None, None)
    # 将每个块的输出的第一个元素链接为下一个 conv_block 的输入。 
    # 每个新步骤将卷积核数量加倍
    cblock2 = conv_block(None, None)
    cblock3 = conv_block(None, None)
    cblock4 = conv_block(None, None, dropout=None) # 为该层添加 0.3 的 dropout_prob
    # 在此层中包含 0.3 的 dropout_prob，并避免 max_pooling 层
    cblock5 = conv_block(None, None, dropout=None, max_pooling=None) 
    ### END CODE HERE
    
    # 扩展路径（解码）
    # 添加第一个 upsampling_block。
    # 使用 cblock5[0] 作为 expansive_input，cblock4[1] 作为 contractive_input 和 n_filters * 8
    ### START CODE HERE
    ublock6 = upsampling_block(None, None,  None)
    # 将前一个区块的输出链接为 expansive_input 和相应的收缩区块输出。
    # 请注意，你必须在 maxpooling 层之前使用收缩块 i.e 的第二个元素。 
    # 在每一步中，使用前一个块的卷积核数量的一半 
    ublock7 = upsampling_block(None, None,  None)
    ublock8 = upsampling_block(None, None,  None)
    ublock9 = upsampling_block(None, None,  None)
    ### END CODE HERE

    conv9 = Conv2D(n_filters,
                 3,
                 activation='relu',
                 padding='same',
                 # 设置 'kernel_initializer' 与上面的练习相同
                 kernel_initializer='he_normal')(ublock9)

    # 添加 Conv2D 层：卷积核数量为 n_classes，卷积核大小为 1，填充方式为 'same'
    ### START CODE HERE
    conv10 = Conv2D(None, None, padding=None)(conv9)
    ### END CODE HERE
    
    model = tf.keras.Model(inputs=inputs, outputs=conv10)

    return model

In [ ]:
import outputs
img_height = 96
img_width = 128
num_channels = 3

unet = unet_model((img_height, img_width, num_channels))
comparator(summary(unet), outputs.unet_model_output)


<a name='3-5'></a>
### 3.5 - 设置模型尺寸

In [ ]:
img_height = 96
img_width = 128
num_channels = 3

unet = unet_model((img_height, img_width, num_channels))

### 查看下面的模型摘要！ 

In [ ]:
unet.summary()

<a name='3-6'></a>
### 3.6 - 损失函数

在语义分割中，你需要与对象类一样多的掩码。在你使用的数据集中，每个掩码中的每个像素都被分配了一个整数概率，它属于某个类别，从 0 到 num_classes-1。正确的类别是概率较高的层。 

这与分类交叉熵不同，其中标签应采用 one-hot 编码（仅 0 和 1）。在这里，你将使用稀疏分类交叉熵作为损失函数来执行逐像素多类预测。当你处理大量类时，稀疏分类交叉熵比其他损失函数更有效。

In [ ]:
unet.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

<a name='3-7'></a>
### 3.7 - 数据集处理

下面，定义一个函数，它允许你显示输入图像及其基本事实：真实掩码。真正的掩码是你训练的模型输出的目标是尽可能接近。 

In [ ]:
def display(display_list):
    plt.figure(figsize=(15, 15))

    title = ['Input Image', 'True Mask', 'Predicted Mask']

    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i+1)
        plt.title(title[i])
        plt.imshow(tf.keras.preprocessing.image.array_to_img(display_list[i]))
        plt.axis('off')
    plt.show()

In [ ]:
for image, mask in image_ds.take(1):
    sample_image, sample_mask = image, mask
    print(mask.shape)
display([sample_image, sample_mask])

In [ ]:
for image, mask in processed_image_ds.take(1):
    sample_image, sample_mask = image, mask
    print(mask.shape)
display([sample_image, sample_mask])

<a name='4'></a>
## 4 - 训练模型

In [ ]:
EPOCHS = 5
VAL_SUBSPLITS = 5
BUFFER_SIZE = 500
BATCH_SIZE = 32
train_dataset = processed_image_ds.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
print(processed_image_ds.element_spec)
model_history = unet.fit(train_dataset, epochs=EPOCHS)

<a name='4-1'></a>
### 4.1 - 创建预测掩码 

现在，定义一个函数，它在类数轴上使用 `tf.argmax` 返回具有最大值的索引，并将预测合并到单个图像中：

In [ ]:
def create_mask(pred_mask):
    pred_mask = tf.argmax(pred_mask, axis=-1)
    pred_mask = pred_mask[..., tf.newaxis]
    return pred_mask[0]

<a name='4-2'></a>
### 4.2 - 绘制模型准确率

让我们看看你的模型表现如何！ 

In [ ]:
plt.plot(model_history.history["accuracy"])

<a name='4-3'></a>
### 4.3 - 显示预测 

接下来，根据真实掩码和原始输入图像检查预测掩码：

In [ ]:
def show_predictions(dataset=None, num=1):
    """
    显示前 num 个批次中每个批次的第一张图像
    """
    if dataset:
        for image, mask in dataset.take(num):
            pred_mask = unet.predict(image)
            display([image[0], mask[0], create_mask(pred_mask)])
    else:
        display([sample_image, sample_mask,
             create_mask(unet.predict(sample_image[tf.newaxis, ...]))])

In [ ]:
show_predictions(train_dataset, 6)

通过 40 个 epoch，你将获得惊人的结果！

**注意**：更多的epoch并不总是能保证好的结果。永远记得去尝试！ 

在Notebook的[底部](#bottom)中，你会找到使用TensorFlow的[enable_op_determinism](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/config/experimental/enable_op_determinism)的训练代码。 “_当启用操作确定性时，TensorFlow 操作将是确定性的。这意味着，如果一个操作在同一硬件上使用相同的输入多次运行，则每次都会有完全相同的输出。_”

## 为其他学习者释放资源

为了给我们的学习者提供流畅的学习体验，请通过运行下面的单元格来释放你的作业使用的资源，以便其他学习者可以像你一样利用这些资源。谢谢你！

**注意**： 
- 完成作业并准备好提交进行评分后，运行下面的单元格。
- 运行它时，将打开一个弹出窗口，单击“`Ok`”。
- 运行单元将是 `restart the kernel`。

In [ ]:
%%javascript
IPython.notebook.save_checkpoint();
if (confirm("Clear memory?") == true)
{
    IPython.notebook.kernel.restart();
}

### 结论 

你已经到了作业的结尾了。非常棒的工作，为语义图像分割创建了最先进的模型！对于自动驾驶汽车来说，这是一项非常重要的任务。埃隆·马斯克肯定会随时敲开你的门。 ;) 

<div style="color:blue;">

**你应该记住什么**： 

* 语义图像分割预测图像中每个像素的标签
* U-Net 使用相同数量的卷积块和转置卷积进行下采样和上采样
* 跳跃连接用于防止 U-Net 中的边界像素信息丢失和过拟合

</div>

<a name='bottom'></a>
### 用于获得一致结果的训练代码：

```Python
EPOCHS = 15
VAL_SUBSPLITS = 5
BUFFER_SIZE = 500
BATCH_SIZE = 32

tf.keras.utils.set_random_seed(1)
tf.config.experimental.enable_op_determinism()
train_dataset = processed_image_ds.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
print(processed_image_ds.element_spec)

unet = unet_model((img_height, img_width, num_channels))
unet.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
model_history = unet.fit(train_dataset, epochs=EPOCHS)

```